In [ ]:
import re
import json
import spacy
import nltk
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
try:
    from nltk.corpus import stopwords

except LookupError:
    nltk.download('stopwords')
    from nltk.corpus import stopwords

In [ ]:
nlp = spacy.load("en_core_web_sm")

json_file_path = "/content/Alfreedabot.conversations.json"
user_keywords = {}

custom_phrases = [
    "thanks", "thank you", "good morning", "good evening", "good afternoon",
    "hello", "hi", "hey", "bye", "okay", "great", "nice", "please", "sure",
    "no problem", "how are you", "hope you are doing well", "take care",
    "welcome", "sorry", "apologies", "yes", "no"
]

In [ ]:
# user = "Baby alFREEda"
user = "User"

In [ ]:
try:
    with open(json_file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    for index, document in enumerate(data):
        messages = document.get("messages", [])
        user_texts = [message.get("content", "") for message in messages if message.get("sender") == user]
        combined_text = " ".join(user_texts)

        for phrase in custom_phrases:
            combined_text = re.sub(r"\b" + re.escape(phrase) + r"\b", "", combined_text, flags=re.IGNORECASE)

        combined_text = re.sub(r"[^\w\s]", "", combined_text)
        combined_text = re.sub(r"\s+", " ", combined_text).strip()

        stop_words = set(stopwords.words("english"))
        tokens = combined_text.split()
        cleaned_text = " ".join([word for word in tokens if word.lower() not in stop_words])

        # vectorizer = TfidfVectorizer()
        # tfidf_matrix = vectorizer.fit_transform([cleaned_text])
        # feature_names = vectorizer.get_feature_names_out()

        # doc = nlp(cleaned_text)
        # keywords = [token.text for token in doc if token.text in feature_names and token.pos_ in {"NOUN", "ADJ"}]

        document_id = document.get("id", f"Document_{index + 1}")
        user_keywords[document_id] = cleaned_text

except (FileNotFoundError, json.JSONDecodeError) as e:
    print(f"Error processing JSON file: {e}")

In [ ]:
all_keywords = "\n".join(["".join(keywords) for keywords in user_keywords.values()])

In [ ]:
wordcloud = WordCloud(
    width = 800,
    height = 400,
    background_color = "white",
    stopwords = set(custom_phrases),
    colormap = "viridis"
).generate(all_keywords)

In [ ]:
words = wordcloud.process_text(all_keywords)
sorted_words = sorted(words.items(), key=lambda x: x[1])
sorted_words.reverse()
sorted_words

In [ ]:
plt.figure(figsize = (10, 5))
plt.imshow(wordcloud, interpolation = "bilinear")
plt.axis("off")
plt.title("Word Cloud of Relevant Keywords", fontsize = 16)
plt.show()